# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nfatima25seecs/ml-pipeline-ex/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one content item, for one client, on one calendar day.**
This is the grain of `fact_content_daily_performance`: the composite key is
`report_date + client_hash_id + content_hash_id`. One row tells me: on this day, for this
client's content item, how many GSC impressions/clicks it got and what its average search
position was - plus whether GA4 engagement data exists for that day.

**Time window:** I develop on the mid-panel month **`month=2026-03`** (2026-03-01 -> 2026-03-31),
as the flyrank-data skill instructs - never on the `_sample` table, which is the panel's final
month (June 2026) and would let my label logic peek at the future outcome window it's supposed
to predict. June 2026 stays a sealed test month for later.

I verify both of these claims with a query below.


In [ ]:
# Environment setup - same pattern as starter notebooks/03_working_with_the_full_release.ipynb
%pip -q install duckdb huggingface_hub

import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste a token in a cell —
# this repo is public. Use a Colab Secret named HF_TOKEN so the getpass prompt never fires.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

# Light sanity check for the unit-of-analysis + time-window claim above.
# COUNT(*) and MIN/MAX(date) touch Parquet metadata, not data — near-free, run this first always.
check = con.sql(f"SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d FROM {MARCH}").df()
print(check)


## 2. Fields: feature / label / context / excluded

**Tables I use:** `fact_content_daily_performance` (month=2026-03 partition, the primary table),
`dim_content` (content metadata, joined for context only), `dim_clients` (per-client history
start dates, joined for context only).

**What I'd predict or rank:** the same task type as w02 — a **scoring/ranking task**. For each
content item, a decline-risk score for March, producing a ranked review queue. Here I build the
label directly from real daily GSC impressions inside the warehouse (second half of March vs
first half), instead of relying on the CSV's precomputed `trend_direction` — that field isn't
shipped in the warehouse, and even if it were, the flyrank-data skill is explicit that it (and
`trend_pct`) may never be a feature since the label is computed from it.

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks` | Feature | Observed daily counts, known the moment that day closes — safe as long as I only sum days *before* the decision point |
| `gsc_avg_position` | Feature | Observed daily position (excluding `0` = no data); same reasoning |
| `is_declining_march` (my own label, built below) | Label / proxy | The thing I'm scoring for — never a feature |
| `report_date`, `client_hash_id`, `content_hash_id` | Context | Grouping/join keys only, pseudonyms with no meaning of their own |
| `url_hash_id`, `keyword_hash_id` (on `dim_content`) | Excluded | Raw-origin hashed context — for dedup/case lookup only, never a feature (they encode a real URL/keyword I never get to see, and reversing that isn't the point of them) |
| `ga4_*` columns when `ga4_data_available IS NOT TRUE` | Excluded | Per the flyrank-data skill, rows before a client's `ga4_data_start` are zero-filled with the flag `FALSE` (and can also be `NULL`) — using the zeros as if they meant "no engagement" would be wrong, so I exclude GA4 features on any row where the flag isn't `TRUE` |

I confirm the schema names actually exist (rather than guess) before building on them:


In [ ]:
# Confirm the columns I'm about to rely on actually exist, with the right names —
# a contract line about a column is a guess until this runs.
cols = con.sql(f"DESCRIBE SELECT * FROM {MARCH} LIMIT 0").df()
print(cols[['column_name', 'column_type']].to_string(index=False))


## 3. Verify it with queries (grain, counts, availability) + five features + the leakage trap

Three small queries on `month=2026-03`, then a five-feature frame for my lane, then the
deliberate-leak experiment.

### 3a. Grain — one row really is what I said

`GROUP BY` the composite key and check nothing repeats. Zero rows back means the grain holds.


In [ ]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows with a duplicate (report_date, client_hash_id, content_hash_id): {len(grain_check)}")
grain_check


### 3b. My slice's row count and date span

My slice for this lane is the full March partition — Lane 2 scores every content item with
March activity, not a narrowed subset.


In [ ]:
slice_facts = con.sql(f"""
    SELECT
        COUNT(*)                          AS n_rows,
        COUNT(DISTINCT client_hash_id)    AS n_clients,
        COUNT(DISTINCT content_hash_id)   AS n_content_items,
        MIN(report_date)                  AS min_date,
        MAX(report_date)                  AS max_date
    FROM {MARCH}
""").df()

slice_facts


### 3c. Availability — filtered with `IS TRUE`

Per the flyrank-data skill, `ga4_data_available` is three-valued (`TRUE` / `FALSE` / `NULL`), so
`= FALSE` or `NOT ga4_data_available` silently mishandles the `NULL` rows. I filter with
`IS TRUE` and show how many rows actually survive for GA4-dependent work.


In [ ]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)  AS ga4_available_rows,
        SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_not_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1)
            AS pct_ga4_available
    FROM {MARCH}
""").df()

availability

### 3d. Five features (max) for my lane — built from March, per content item

Each feature is aggregated **per `(client_hash_id, content_hash_id)`** over March. Decision
moment = end of March (2026-03-31), the point at which a reviewer would look at this score.

1. **`impressions_march`** — total GSC impressions across all of March.
   *Knowable at the decision moment because* every day it sums already happened by 2026-03-31.
2. **`clicks_march`** — total GSC clicks across all of March.
   *Knowable at the decision moment because* same reasoning — only past days are summed.
3. **`avg_position_march`** — average GSC position across March (excluding `gsc_avg_position = 0`,
   which the dictionary defines as "no data", not rank zero).
   *Knowable at the decision moment because* it's a plain average of daily positions that already
   happened.
4. **`days_with_impressions_march`** — count of distinct days in March with ≥1 impression (a
   coverage/consistency signal, 0–31).
   *Knowable at the decision moment because* it only counts days that have already closed.
5. **`impressions_first_half_march`** — impressions summed over 2026-03-01 → 2026-03-15 only.
   *Knowable at the decision moment because* it's a strict subset of the past, and — importantly —
   it does **not** touch the second half of the month, which is what my label (below) is built
   from. Keeping this window boundary explicit is what keeps feature #5 honest and feature #6
   (next) dishonest.


In [ ]:
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                                  AS impressions_march,
        SUM(gsc_clicks)                                                       AS clicks_march,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)         AS avg_position_march,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END)    AS days_with_impressions_march,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END)
                                                                               AS impressions_first_half_march,
        -- kept separate on purpose, NOT yet joined into `features` as a model feature —
        -- this is the ingredient the label (and the trap) is built from:
        SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END)
                                                                               AS impressions_second_half_march
    FROM {MARCH}
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

# The label: declining this month, mirrors the CSV's trend_direction == 'down' logic,
# but built directly from real daily impressions instead of a precomputed field.
features['is_declining_march'] = (
    features['impressions_second_half_march'] < features['impressions_first_half_march']
).astype(int)

print(f"{len(features):,} content items with March impressions")
print(f"Declining share: {features['is_declining_march'].mean():.1%}")
features.head()


### 3e. The trap — one label-derived column, on purpose

`is_declining_march` is defined as *second-half impressions < first-half impressions*. If I now
add `impressions_second_half_march` itself as a model feature, I've handed the model one side of
its own label's comparison - that's leakage, the same lesson notebook 02 taught with
`trend_direction` / `trend_pct`. I expect the score to jump toward a suspiciously perfect number.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

honest_cols = ['impressions_march', 'clicks_march', 'avg_position_march',
               'days_with_impressions_march', 'impressions_first_half_march']
leaky_cols  = honest_cols + ['impressions_second_half_march']

model_data = features.dropna(subset=leaky_cols).copy()
y = model_data['is_declining_march']

X_tr_idx, X_te_idx = train_test_split(
    model_data.index, test_size=0.25, random_state=42, stratify=y
)

def quick_auc(cols):
    X_tr, X_te = model_data.loc[X_tr_idx, cols], model_data.loc[X_te_idx, cols]
    y_tr, y_te = y.loc[X_tr_idx], y.loc[X_te_idx]
    clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    proba = clf.predict_proba(X_te)[:, 1]
    return roc_auc_score(y_te, proba)

leaky_auc  = quick_auc(leaky_cols)
honest_auc = quick_auc(honest_cols)

print(f"WITH the leak (impressions_second_half_march included):  ROC-AUC = {leaky_auc:.3f}")
print(f"Deleting the leak, honest features only:                  ROC-AUC = {honest_auc:.3f}")
print()
print("The leaky score jumps toward 1.0 because the label IS a comparison against that column —")
print("the model isn't learning a pattern, it's reading the answer key. The honest number above,")
print("built only from impressions_first_half_march and pre-decision-point aggregates, is the real one.")


## 4. Data limits

**Named limitation: this is an unbalanced panel, and my March slice inherits that.** Per-client
history depth differs wildly - some clients have 17 months of data, some far less, and
`dim_clients.gsc_data_start` / `ga4_data_start` are the only honest way to know which. A content
item that looks "new" or "low-volume" in March might simply belong to a client whose tracking
started that month, not a client whose content is actually thin. My grain and availability checks
above are for March 2026 only - they say nothing about whether this same row count, decline
share, or GA4 coverage would hold in a different month, and I have not checked that here.


In [ ]:
# Optional supporting check for the limitation above: how many clients have less than
# a full month of GSC history *before* March even starts (i.e., gsc_data_start falls inside
# or after my March window) - a quick way to see the unbalanced panel touching this slice.
coverage = con.sql(f"""
    SELECT
        COUNT(*) AS n_clients,
        SUM(CASE WHEN gsc_data_start >= DATE '2026-03-01' THEN 1 ELSE 0 END) AS clients_new_in_or_after_march
    FROM {DIM_CLIENTS}
""").df()

coverage


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.